In [1]:
! nvidia-smi


Mon Oct  5 17:31:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install transformers[sentencepiece] datasets sacrebleu rouge_score py7zr -q

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.2/72.2 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 492.7/492.7 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/52.4 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.5/144.5 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 10.3 MB/s eta 0:00:00


In [3]:
!pip install --upgrade accelerate
!pip install -y transformers accelerate
!pip install transformers accelerate


Usage:   
  pip3 install [options] <requirement specifier> [package-index-options] ...
  pip3 install [options] -r <requirements file> [package-index-options] ...
  pip3 install [options] [-e] <vcs project url> ...
  pip3 install [options] [-e] <local project path> ...
  pip3 install [options] <archive url/path> ...

no such option: -y


In [4]:
from transformers import pipeline , set_seed
from datasets import load_dataset,load_from_disk

import matplotlib.pyplot as plt
import pandas as pd
from transformers import AutoModelForSeq2SeqLM,AutoTokenizer,TrainingArguments
# from datasets import load_dataset,load_metric
import nltk
from nltk.tokenize import sent_tokenize
from tqdm import tqdm
import torch

nltk.download("punkt")





[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [6]:
device="cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
device="cuda" if torch.cuda.is_available() else "cpu"

model_ckpt="google/pegasus-cnn_dailymail"

tokenizer=AutoTokenizer.from_pretrained(model_ckpt)
# Tokenizer ise model ke samajhne layak numerical representation mein convert karta hai.
# from_pretrained(model_ckpt) ka matlab:
# Jo tokenizer google/pegasus-cnn_dailymail model ke saath use hota hai, wahi download/load karo.
model_pegasus=AutoModelForSeq2SeqLM.from_pretrained(model_ckpt).to(device)#here we actually load the model and sent to device


In [7]:
# !wget https://github.com/entbappy/Branching-tutorial/blob/master/summarizer-data.zip
!unzip summarizer-data.zip

Archive:  summarizer-data.zip
  inflating: samsum-test.csv         
  inflating: samsum-train.csv        
  inflating: samsum-validation.csv   
   creating: samsum_dataset/
 extracting: samsum_dataset/dataset_dict.json  
   creating: samsum_dataset/test/
  inflating: samsum_dataset/test/data-00000-of-00001.arrow  
  inflating: samsum_dataset/test/dataset_info.json  
  inflating: samsum_dataset/test/state.json  
   creating: samsum_dataset/train/
  inflating: samsum_dataset/train/data-00000-of-00001.arrow  
  inflating: samsum_dataset/train/dataset_info.json  
  inflating: samsum_dataset/train/state.json  
   creating: samsum_dataset/validation/
  inflating: samsum_dataset/validation/data-00000-of-00001.arrow  
  inflating: samsum_dataset/validation/dataset_info.json  
  inflating: samsum_dataset/validation/state.json  


In [8]:
data_samsum=load_from_disk('samsum_dataset')
data_samsum

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 14732
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 819
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 818
    })
})

In [9]:
split_length=[len(data_samsum[split]) for split in data_samsum]

print(f"split lengths: {split_length}")

print(f"features: {data_samsum["train"].column_names}")
print(f"dialogue")
print(f"features: {data_samsum["test"][1]["dialogue"]}")
print(f"summary ")
print(f"features: {data_samsum["test"][1]["summary"]}")

split lengths: [14732, 819, 818]
features: ['id', 'dialogue', 'summary']
dialogue
features: Eric: MACHINE!
Rob: That's so gr8!
Eric: I know! And shows how Americans see Russian ;)
Rob: And it's really funny!
Eric: I know! I especially like the train part!
Rob: Hahaha! No one talks to the machine like that!
Eric: Is this his only stand-up?
Rob: Idk. I'll check.
Eric: Sure.
Rob: Turns out no! There are some of his stand-ups on youtube.
Eric: Gr8! I'll watch them now!
Rob: Me too!
Eric: MACHINE!
Rob: MACHINE!
Eric: TTYL?
Rob: Sure :)
summary 
features: Eric and Rob are going to watch a stand-up on youtube.


In [10]:
def convert_examples_to_features(example_batch):
  input_encodings=tokenizer(example_batch["dialogue"],max_length=1024,truncation=True)
  target_encodings=tokenizer(example_batch["summary"],max_length=128,truncation=True)

  return {
      "input_ids": input_encodings['input_ids'],
      "attention_mask": input_encodings["attention_mask"],
      "labels": target_encodings['input_ids']
  }

In [11]:
dataet_samsum_pt=data_samsum.map(convert_examples_to_features,batched=True)

Map:   0%|          | 0/14732 [00:00<?, ? examples/s]

Map:   0%|          | 0/819 [00:00<?, ? examples/s]

Map:   0%|          | 0/818 [00:00<?, ? examples/s]

In [12]:
dataet_samsum_pt["train"]

Dataset({
    features: ['id', 'dialogue', 'summary', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 14732
})

In [13]:
from transformers import DataCollatorForSeq2Seq#prepare the tokens for trainning batch
seq2seq_data_collator=DataCollatorForSeq2Seq(tokenizer,model=model_pegasus)

In [14]:
trainer_args = TrainingArguments(
    output_dir="pegasus-samsum",
    num_train_epochs=3,#poora trainnig dataset model ko kitni baar dikhana h

    per_device_train_batch_size=1,#ek time par model ko kitne trainning example dene h
    per_device_eval_batch_size=1,#same like for train but this for eval

    learning_rate=2e-5,
    weight_decay=0.01,#help to protect from overfitting
    warmup_steps=500,#500 steps tkk lr gradually increase krega ek dum nhi bdate

    logging_steps=10,#kitne steps baad trainning ki info print krni h

    eval_strategy="steps",
    eval_steps=500,

    save_strategy="epoch",
    save_total_limit=1
)

In [15]:
from transformers import Trainer
trainer=Trainer(model=model_pegasus,args=trainer_args,processing_class=tokenizer,
                data_collator=seq2seq_data_collator,train_dataset=dataet_samsum_pt['train'],
                eval_dataset=dataet_samsum_pt['validation'])

In [ ]:
# trainer.train(
#     resume_from_checkpoint="/content/pegasus-samsum/checkpoint-3500"
# )

In [ ]:
import os

for root, dirs, files in os.walk("pegasus-samsum"):
    for file in files:
        path = os.path.join(root, file)
        print(path)

In [16]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [17]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.1 MB/s eta 0:00:00


In [18]:
def generate_batch_sized_chunks(list_of_elements, batch_size):
    """Split the dataset into smaller batches."""
    for i in range(0, len(list_of_elements), batch_size):
        yield list_of_elements[i:i + batch_size]#yield ek-ek batch dheere-dheere return karta hai instead of saari list ek saath.



# Test dataset ke dialogues ko batches mein lekar trained PEGASUS se summaries generate karna,
# un generated summaries ko actual/original summaries ke saath compare karna, aur end mein ROUGE score nikalna.
def calculate_metric_on_test_ds(
    dataset,
    metric,
    model,
    tokenizer,
    batch_size=16,
    device=device,
    column_text="article",
    column_summary="highlights"
):

    article_batches = list(
        generate_batch_sized_chunks(dataset[column_text], batch_size)
    )

    target_batches = list(
        generate_batch_sized_chunks(dataset[column_summary], batch_size)
    )

    for article_batch, target_batch in tqdm(# tqdm progress bar dikhata hai..
        zip(article_batches, target_batches),#making pair of dialogue and summary
        total=len(article_batches)#tqdm ko btata h  ki kitne batches h
    ):

        inputs = tokenizer(
            article_batch,
            max_length=1024,
            truncation=True,
            padding="max_length",
            return_tensors="pt"#tokenize ouptut pytorch tensor mein dega instead of python list.
        )

        summaries = model.generate(
            input_ids=inputs["input_ids"].to(device),#input ids genrated tokens h tokenizer ne jo kre h.
            attention_mask=inputs["attention_mask"].to(device),#to define which is real tokens and which is padding number
            length_penalty=0.8,
            num_beams=8,#Ye beam search hai.
                        # Model ek hi possible summary generate karne ke bajaye multiple possible sequences explore karta hai.
            max_length=128
        )

        # Decode generated summaries
        #basically abb summaries mein tokens h toh use text mein convert krne ke liye
        decode_summaries = [
            tokenizer.decode(
                s,
                skip_special_tokens=True,
                clean_up_tokenization_space=True
            )
            for s in summaries
        ]

        # Add predictions and actual summaries
        metric.add_batch(
            predictions=decode_summaries,
            references=target_batch
        )

    # Calculate final ROUGE score
    score = metric.compute()

    return score

In [25]:
# evaluate Hugging Face ki library hai jo model ki performance measure
#  karne ke liye metrics provide karti hai.

import evaluate
rouge_names=["rouge1","rouge2","rougeL","rougeLsum"]

rouge_metric = evaluate.load("rouge")#ROUGE calculate karne ki functionality aa jaati hai.

In [20]:
#load the model
model_path = "/content/drive/MyDrive/pegasus-samsum/checkpoint-44196"

tokenizer = AutoTokenizer.from_pretrained(model_path)

model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(model_path)

model_pegasus = model_pegasus.to(device)

Loading weights:   0%|          | 0/680 [00:00<?, ?it/s]

In [21]:
score = calculate_metric_on_test_ds(
    data_samsum["test"][0:10],
    rouge_metric,
    model_pegasus,
    tokenizer,
    batch_size=2,
    column_text="dialogue",
    column_summary="summary"
)

100%|██████████| 5/5 [00:23<00:00,  4.66s/it]


In [22]:
rouge_dict=dict((rn,score[rn]) for rn in rouge_names)

pd.DataFrame(rouge_dict,index=["pegasus"])

,rouge1,rouge2,rougeL,rougeLsum
pegasus,0.438449,0.233243,0.344888,0.345619


In [23]:
from transformers import pipeline
gen_kwargs={"length_penalty": 0.8, "num_beams": 8, "max_length": 128}

sample_text=data_samsum["test"][0]["dialogue"]
reference=data_samsum["test"][0]["summary"]

pipe = pipeline("summarization", model=model_pegasus,tokenizer=tokenizer)

print("Dialogue")
print(sample_text)
print("reference summary")
print(reference)
print("model summary")
print(pipe(sample_text, **gen_kwargs)[0]["summary_text"])

KeyError: "Unknown task summarization, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection']"

In [24]:
dialogue = data_samsum["test"][0]["dialogue"]
reference = data_samsum["test"][0]["summary"]

inputs = tokenizer(
    dialogue,
    return_tensors="pt",
    max_length=1024,
    truncation=True
).to(device)

summary_ids = model_pegasus.generate(
    **inputs,
    max_length=128,
    num_beams=8,
    length_penalty=0.8
)

generated_summary = tokenizer.decode(
    summary_ids[0],
    skip_special_tokens=True
)

print("Dialogue:")
print(dialogue)

print("\nGenerated Summary:")
print(generated_summary)

print("\nOriginal Summary:")
print(reference)

Dialogue:
Hannah: Hey, do you have Betty's number?
Amanda: Lemme check
Hannah: <file_gif>
Amanda: Sorry, can't find it.
Amanda: Ask Larry
Amanda: He called her last time we were at the park together
Hannah: I don't know him well
Hannah: <file_gif>
Amanda: Don't be shy, he's very nice
Hannah: If you say so..
Hannah: I'd rather you texted him
Amanda: Just text him 🙂
Hannah: Urgh.. Alright
Hannah: Bye
Amanda: Bye bye

Generated Summary:
Amanda can't find Betty's number. Larry called Betty the last time they were at the park together. Hannah wants Amanda to text Larry. 

Original Summary:
Hannah needs Betty's number but Amanda doesn't have it. She needs to contact Larry.


In [26]:
import inspect#ye kissi bhi function ke by default argument ko pta krne ke liye..

print(inspect.signature(model_pegasus.generate))

(inputs: torch.Tensor | None = None, generation_config: transformers.generation.configuration_utils.GenerationConfig | None = None, logits_processor: transformers.generation.logits_process.LogitsProcessorList | None = None, stopping_criteria: transformers.generation.stopping_criteria.StoppingCriteriaList | None = None, prefix_allowed_tokens_fn: collections.abc.Callable[[int, torch.Tensor], list[int]] | None = None, synced_gpus: bool | None = None, assistant_model: Optional[ForwardRef('PreTrainedModel')] = None, streamer: Optional[ForwardRef('BaseStreamer')] = None, negative_prompt_ids: torch.Tensor | None = None, negative_prompt_attention_mask: torch.Tensor | None = None, custom_generate: str | collections.abc.Callable | None = None, **kwargs) -> transformers.generation.utils.GenerateDecoderOnlyOutput | transformers.generation.utils.GenerateEncoderDecoderOutput | transformers.generation.utils.GenerateBeamDecoderOnlyOutput | transformers.generation.utils.GenerateBeamEncoderDecoderOutput